# ETL Notebook
---
Running this notebook from end-to-end will reconstruct all tables from the raw sources

In [110]:
"""Run this to establish the connection with the database"""
# load the SQL extension
%load_ext sql

# create the connection string
import os
user = os.getenv('postgres_user')
password = os.getenv('postgres_password')
db = os.getenv('postgres_db')
db_url = f"postgresql://{user}:{password}@db/{db}"

# establish the connection
from sqlalchemy import create_engine
engine = create_engine(db_url)

# set %sql magic commands to use the connection
%sql engine

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [60]:
%%sql
-- Create states table from raw table
DROP TABLE states;

CREATE TABLE
  IF NOT EXISTS states (
    id VARCHAR(2) PRIMARY KEY,
    fp VARCHAR(2),
    name VARCHAR(255),
    code VARCHAR(2),
    division_id VARCHAR(1),
    region_id VARCHAR(1),
    geometry GEOMETRY (MULTIPOLYGON, 4326)
  );

INSERT INTO
  states (
    id,
    fp,
    name,
    code,
    division_id,
    region_id,
    geometry
  )
SELECT
  "GEOID" AS id,
  "STATEFP" AS fp,
  UPPER("NAME") AS name,
  "STUSPS" AS code,
  "DIVISION" AS division_id,
  "REGION" AS region_id,
  geometry
FROM
  "census_2024_state_raw"

Running query in 'postgresql://healthgps:***@db/healthgps'

56 rows affected.

++
||
++
++

In [64]:
%%sql
-- Create counties table from raw table
DROP TABLE counties;

CREATE TABLE
  counties (
    id VARCHAR(5) PRIMARY KEY,
    fp VARCHAR(3),
    name VARCHAR(255),
    state_name VARCHAR(255),
    state_code VARCHAR(2),
    state_id VARCHAR(2),
    division_id VARCHAR(1),
    region_id VARCHAR(1),
    geometry GEOMETRY (MULTIPOLYGON, 4326)
  );

INSERT INTO
  counties (
    id,
    fp,
    name,
    state_name,
    state_code,
    state_id,
    division_id,
    region_id,
    geometry
  )
SELECT
  "GEOID" AS id,
  "COUNTYFP" AS fp,
  UPPER("NAME") AS name,
  states.name AS state_name,
  states.code AS state_code,
  states.id AS state_id,
  states.division_id,
  states.region_id,
  counties.geometry AS geometry
FROM
  "census_2024_county_raw" counties
  LEFT JOIN states ON states.fp = counties."STATEFP"

Running query in 'postgresql://healthgps:***@db/healthgps'

3235 rows affected.

++
||
++
++

In [65]:
%%sql
-- Create tracts table from raw table
DROP TABLE tracts;

CREATE TABLE
  tracts (
    id VARCHAR(11) PRIMARY KEY,
    fp VARCHAR(6),
    name VARCHAR(255),
    county_name VARCHAR(255),
    state_name VARCHAR(255),
    state_code VARCHAR(2),
    county_id VARCHAR(5),
    state_id VARCHAR(2),
    division_id VARCHAR(1),
    region_id VARCHAR(1),
    geometry GEOMETRY (MULTIPOLYGON, 4326)
  );

INSERT INTO
  tracts (
    id,
    fp,
    name,
    county_name,
    state_name,
    state_code,
    county_id,
    state_id,
    division_id,
    region_id,
    geometry
  )
SELECT
  "GEOID" AS id,
  "TRACTCE" AS fp,
  UPPER("NAME") AS name,
  counties.name AS county_name,
  counties.state_name AS state_name,
  counties.state_code AS state_code,
  counties.id AS county_id,
  counties.state_id AS state_id,
  counties.division_id AS division_id,
  counties.region_id AS region_id,
  tracts.geometry AS geometry
FROM
  "census_2024_tract_raw" tracts
  LEFT JOIN counties ON counties.id = tracts."STATEFP" || tracts."COUNTYFP"

Running query in 'postgresql://healthgps:***@db/healthgps'

85529 rows affected.

++
||
++
++

In [66]:
%%sql
-- Create block groups table from raw table
DROP TABLE blkgrps;

CREATE TABLE
  blkgrps (
    id VARCHAR(12) PRIMARY KEY,
    fp VARCHAR(1),
    name VARCHAR(255),
    tract_name VARCHAR(255),
    county_name VARCHAR(255),
    state_name VARCHAR(255),
    state_code VARCHAR(2),
    tract_id VARCHAR(11),
    county_id VARCHAR(5),
    state_id VARCHAR(2),
    division_id VARCHAR(1),
    region_id VARCHAR(1),
    geometry GEOMETRY (MULTIPOLYGON, 4326)
  );

INSERT INTO
  blkgrps (
    id,
    fp,
    name,
    tract_name,
    county_name,
    state_name,
    state_code,
    tract_id,
    county_id,
    state_id,
    division_id,
    region_id,
    geometry
  )
SELECT
  "GEOID" AS id,
  "BLKGRPCE" AS fp,
  UPPER("NAMELSAD") AS name,
  tracts.name AS tract_name,
  tracts.county_name AS county_name,
  tracts.state_name AS state_name,
  tracts.state_code AS state_code,
  tracts.id AS tract_id,
  tracts.county_id AS county_id,
  tracts.state_id AS state_id,
  tracts.division_id AS division_id,
  tracts.region_id AS region_id,
  blocks.geometry AS geometry
FROM
  "census_2024_blkgrp_raw" blocks
  LEFT JOIN tracts ON tracts.id = blocks."STATEFP" || blocks."COUNTYFP" || blocks."TRACTCE"

Running query in 'postgresql://healthgps:***@db/healthgps'

242748 rows affected.

++
||
++
++

In [67]:
%%sql
-- Add unique identifiers for each pantry row for tracing
ALTER TABLE access_food_foodbanks_raw
ADD COLUMN traceid UUID DEFAULT (gen_random_uuid ());

ALTER TABLE snap_retailers_raw
ADD COLUMN traceid UUID DEFAULT (gen_random_uuid ());

ALTER TABLE why_hungry_foodbanks_raw
ADD COLUMN traceid UUID DEFAULT (gen_random_uuid ());

ALTER TABLE capital_food_bank_raw
ADD COLUMN traceid UUID DEFAULT (gen_random_uuid ());

ALTER TABLE md_food_bank_raw
ADD COLUMN traceid UUID DEFAULT (gen_random_uuid ());

Running query in 'postgresql://healthgps:***@db/healthgps'

RuntimeError: (psycopg2.errors.DuplicateColumn) column "traceid" of relation "access_food_foodbanks_raw" already exists

[SQL: ALTER TABLE access_food_foodbanks_raw
ADD COLUMN traceid UUID DEFAULT (gen_random_uuid ());]
(Background on this error at: https://sqlalche.me/e/20/f405)


In [33]:
%%sql
-- Create foodbanks table
DROP TABLE food_outposts CASCADE;

CREATE TABLE
  food_outposts (
    id UUID PRIMARY KEY,
    source VARCHAR(255),
    alt_id VARCHAR(255),
    name VARCHAR(510),
    address VARCHAR(510),
    phone_number_1 VARCHAR(255),
    phone_number_2 VARCHAR(255),
    type VARCHAR(255),
    blkgrp_name VARCHAR(255),
    tract_name VARCHAR(255),
    county_name VARCHAR(255),
    state_name VARCHAR(255),
    state_code VARCHAR(2),
    blkgrp_id VARCHAR(12),
    tract_id VARCHAR(11),
    county_id VARCHAR(5),
    state_id VARCHAR(2),
    division_id VARCHAR(1),
    region_id VARCHAR(1),
    duplicate BOOLEAN,
    geometry GEOMETRY (POINT, 4326)
  );

CREATE INDEX food_outposts_geom_idx ON food_outposts USING GIST (geometry);

Running query in 'postgresql://healthgps:***@db/healthgps'

++
||
++
++

In [34]:
%%sql
-- Maryland banks
INSERT INTO
  food_outposts (
    id,
    source,
    alt_id,
    name,
    address,
    phone_number_1,
    phone_number_2,
    type,
    blkgrp_name,
    tract_name,
    county_name,
    state_name,
    state_code,
    blkgrp_id,
    tract_id,
    county_id,
    state_id,
    division_id,
    region_id,
    duplicate,
    geometry
  )
SELECT
  traceid AS id,
  'md_food_bank' AS source,
  foodbank."ID" AS alt_id,
  UPPER(foodbank.title) AS name,
  NULL AS address,
  NULL AS phone_number_1,
  NULL AS phone_number_2,
  'FOOD PANTRY' AS type,
  blkgrps.name AS blkgrp_name,
  blkgrps.tract_name AS tract_name,
  blkgrps.county_name AS county_name,
  blkgrps.state_name AS state_name,
  blkgrps.state_code AS state_code,
  blkgrps.id AS blkgrp_id,
  blkgrps.tract_id AS tract_id,
  blkgrps.county_id AS county_id,
  blkgrps.state_id AS state_id,
  blkgrps.division_id AS division_id,
  blkgrps.region_id AS region_id,
  False as duplicate,
  foodbank.geometry
FROM
  blkgrps
  INNER JOIN md_food_bank_raw foodbank ON st_contains (blkgrps.geometry, foodbank.geometry)

Running query in 'postgresql://healthgps:***@db/healthgps'

252 rows affected.

++
||
++
++

In [35]:
%%sql
-- DC banks
INSERT INTO
  food_outposts (
    id,
    source,
    alt_id,
    name,
    address,
    phone_number_1,
    phone_number_2,
    type,
    blkgrp_name,
    tract_name,
    county_name,
    state_name,
    state_code,
    blkgrp_id,
    tract_id,
    county_id,
    state_id,
    division_id,
    region_id,
    duplicate,
    geometry
  )
SELECT
  traceid AS id,
  'capital_food_bank' AS source,
  foodbank.agency_ref AS alt_id,
  UPPER(foodbank.name),
  UPPER(concat_ws (', ', address1, city, state, zip)) AS address,
  replace (phone, ' ', '') AS phone_number_1,
  NULL AS phone_number_2,
  'FOOD PANTRY' AS type,
  blkgrps.name AS blkgrp_name,
  blkgrps.tract_name AS tract_name,
  blkgrps.county_name AS county_name,
  blkgrps.state_name AS state_name,
  blkgrps.state_code AS state_code,
  -- blkgrp.id as blkgrp_id,
  -- blkgrp.tract_id as tract_id,
  blkgrps.id AS blkgrp_id,
  blkgrps.tract_id AS tract_id,
  blkgrps.county_id AS county_id,
  blkgrps.state_id AS state_id,
  blkgrps.division_id AS division_id,
  blkgrps.region_id AS region_id,
  False as duplicate,
  foodbank.geometry
FROM
  blkgrps
  INNER JOIN capital_food_bank_raw foodbank ON st_contains (blkgrps.geometry, foodbank.geometry)

Running query in 'postgresql://healthgps:***@db/healthgps'

368 rows affected.

++
||
++
++

In [36]:
%%sql
-- WhyHunger banks
INSERT INTO
  food_outposts (
    id,
    source,
    alt_id,
    name,
    address,
    phone_number_1,
    phone_number_2,
    type,
    blkgrp_name,
    tract_name,
    county_name,
    state_name,
    state_code,
    blkgrp_id,
    tract_id,
    county_id,
    state_id,
    division_id,
    region_id,
    duplicate,
    geometry
  )
SELECT
  traceid AS id,
  'why_hunger_api' AS source,
  foodbank.id AS alt_id,
  UPPER(foodbank.name),
  replace (
    replace (replace (UPPER("Address"), ',,', ','), ', ', ','),
    ',',
    ', '
  ) AS address,
  replace ("Phones", ' ', '') AS phone_number_1,
  NULL AS phone_number_2,
  UPPER(type),
  blkgrps.name AS blkgrp_name,
  blkgrps.tract_name AS tract_name,
  blkgrps.county_name AS county_name,
  blkgrps.state_name AS state_name,
  blkgrps.state_code AS state_code,
  blkgrps.id AS blkgrp_id,
  blkgrps.tract_id AS tract_id,
  blkgrps.county_id AS county_id,
  blkgrps.state_id AS state_id,
  blkgrps.division_id AS division_id,
  blkgrps.region_id AS region_id,
  False as duplicate,
  foodbank.geometry
FROM
  blkgrps
  INNER JOIN why_hungry_foodbanks_raw foodbank ON st_contains (blkgrps.geometry, foodbank.geometry)

Running query in 'postgresql://healthgps:***@db/healthgps'

37941 rows affected.

++
||
++
++

In [37]:
%%sql
-- Access Food banks
INSERT INTO
  food_outposts (
    id,
    source,
    alt_id,
    name,
    address,
    phone_number_1,
    phone_number_2,
    type,
    blkgrp_name,
    tract_name,
    county_name,
    state_name,
    state_code,
    blkgrp_id,
    tract_id,
    county_id,
    state_id,
    division_id,
    region_id,
    duplicate,
    geometry
  )
SELECT
  traceid AS id,
  'access_food_api' AS source,
  "locationId" AS alt_id,
  UPPER("locationName") AS name,
  UPPER(
    concat_ws (
      ', ',
      address1,
      city,
      state,
      concat (' ', "zipCode", country)
    )
  ) AS address,
  CASE
    WHEN "phoneExt" IS NOT NULL THEN concat_ws ('x', replace (phone, ' ', ''), "phoneExt")
    ELSE replace (phone, ' ', '')
  END AS phone_number_1,
  CASE
    WHEN "contactPhoneExt" IS NOT NULL THEN concat (
      'x',
      replace ("contactPhone", ' ', ''),
      "contactPhoneExt"
    )
    ELSE replace ("contactPhone", ' ', '')
  END AS phone_number_2,
  UPPER("foodPrograms") AS "type",
  blkgrps.name AS blkgrp_name,
  blkgrps.tract_name AS tract_name,
  blkgrps.county_name AS county_name,
  blkgrps.state_name AS state_name,
  blkgrps.state_code AS state_code,
  blkgrps.id AS blkgrp_id,
  blkgrps.tract_id AS tract_id,
  blkgrps.county_id AS county_id,
  blkgrps.state_id AS state_id,
  blkgrps.division_id AS division_id,
  blkgrps.region_id AS region_id,
  False as duplicate,
  foodbank.geometry
FROM
  blkgrps
  INNER JOIN access_food_foodbanks_raw foodbank ON st_contains (blkgrps.geometry, foodbank.geometry)

Running query in 'postgresql://healthgps:***@db/healthgps'

13171 rows affected.

++
||
++
++

In [38]:
%%sql
-- Create clusters from duplicate records
DROP TABLE food_outposts_clusters;

CREATE TABLE
  food_outposts_clusters (
    id UUID PRIMARY KEY,
    source VARCHAR(255),
    alt_id VARCHAR(512),
    name VARCHAR(1024),
    address VARCHAR(1024),
    phone_number_1 VARCHAR(512),
    phone_number_2 VARCHAR(512),
    type VARCHAR(512),
    blkgrp_name VARCHAR(255),
    tract_name VARCHAR(255),
    county_name VARCHAR(255),
    state_name VARCHAR(255),
    state_code VARCHAR(2),
    blkgrp_id VARCHAR(12),
    tract_id VARCHAR(11),
    county_id VARCHAR(5),
    state_id VARCHAR(2),
    division_id VARCHAR(1),
    region_id VARCHAR(1),
    duplicate BOOLEAN,
    geometry GEOMETRY (POINT, 4326)
  );

insert into food_outposts_clusters (
    id,
    source,
    alt_id,
    name,
    address,
    phone_number_1,
    phone_number_2,
    type,
    blkgrp_name,
    tract_name,
    county_name,
    state_name,
    state_code,
    blkgrp_id,
    tract_id,
    county_id,
    state_id,
    division_id,
    region_id,
    duplicate,
    geometry
)
with knn as (
    select
        a.id as a_id,
        b.id as b_id,
        a.name as a_name,
        b.name as b_name,
        a.address as a_address,
        b.address as b_address,
        a.phone_number_1 as a_pn_1,
        b.phone_number_1 as b_pn_1,
        a.phone_number_2 as a_pn_2,
        b.phone_number_2 as b_pn_2,
        a.type as a_type,
        b.type as b_type,
        a.source as a_source,
        b.source as b_source,
        a.geometry as a_geometry,
        b.geometry as b_geometry,
        st_distance(a.geometry::geography, b.geometry::geography) as dist
    from food_outposts a
    cross join lateral (
        select
            id,
            name,
            address,
            phone_number_1,
            phone_number_2,
            type,
            source,
            geometry
        from food_outposts f
        where f.id <> a.id
        order by f.geometry <-> a.geometry
        limit 5
    ) b
),
clusters as (
    SELECT
        a_id,
        ARRAY(SELECT element FROM unnest(array_cat(array[a_id], array_agg(b_id))) AS element ORDER BY element) as cluster_ids,
        ARRAY(SELECT name FROM unnest(array_cat(array[a_name], array_agg(b_name))) AS name ORDER BY name) as cluster_names,
        ARRAY(SELECT similarity(a_name, name) FROM unnest(array_cat(array[a_name], array_agg(b_name))) AS name) as cluster_similarity_names,
        ARRAY(SELECT address FROM unnest(array_cat(array[a_address], array_agg(b_address))) AS address ORDER BY address) as cluster_addresses,
        ARRAY(SELECT pn_1 FROM unnest(array_cat(array[a_pn_1], array_agg(b_pn_1))) AS pn_1 order by pn_1) as cluster_phone_numbers_1,
        ARRAY(SELECT pn_2 FROM unnest(array_cat(array[a_pn_2], array_agg(b_pn_2))) AS pn_2 order by pn_2) as cluster_phone_numbers_2,
        ARRAY(SELECT type FROM unnest(array_cat(array[a_type], array_agg(b_type))) AS type order by type) as cluster_types,
        ARRAY(SELECT geo FROM unnest(array_cat(array[a_geometry], array_agg(b_geometry))) AS geo order by geo) as cluster_geometries,
        ARRAY(SELECT st_distance(a_geometry::geography,geo::geography) FROM unnest(array_cat(array[a_geometry], array_agg(b_geometry))) AS geo) as cluster_distances
    FROM
        knn
    WHERE
        dist < 100
        AND similarity(a_name, b_name) > 0.5
        AND split_part(replace(a_address, ',', ' '), ' ', 1) = split_part(translate(b_address, ',', ' '), ' ', 1)
        -- AND (
        --     translate(a_pn_1, '-() ', '') = translate(b_pn_1, '-() ', '')
        --     OR
        --     translate(a_pn_1, '-() ', '') = translate(b_pn_2, '-() ', '')
        -- )
    GROUP BY a_id, a_name, a_address, a_pn_1, a_pn_2, a_geometry, a_type
),
distinct_clusters AS (
    SELECT
        cluster_ids AS cluster_id,
        cluster_names,
        cluster_addresses,
        cluster_phone_numbers_1,
        cluster_phone_numbers_2,
        cluster_types,
        cluster_geometries
    FROM clusters
    GROUP BY 1,2,3,4,5,6,7
),
superset_clusters AS (
    SELECT
        a.cluster_id AS cluster_id,
        b.cluster_id AS parent_cluster_id
    FROM distinct_clusters a
    CROSS JOIN LATERAL (
        SELECT
            cluster_id
        FROM distinct_clusters b
        WHERE a.cluster_id <@ b.cluster_id
        ORDER BY cardinality(b.cluster_id) DESC
        LIMIT 1
    ) b
),
deduped_clusters AS (
    SELECT
        distinct parent_cluster_id AS cluster_id
    FROM superset_clusters
)
SELECT
    gen_random_uuid() as id,
    'cluster' as source,
    array_to_string(deduped_clusters.cluster_id, '|') as alt_id,
    array_to_string(distinct_clusters.cluster_names, '|') as name,
    array_to_string(distinct_clusters.cluster_addresses, '|') as address,
    array_to_string(distinct_clusters.cluster_phone_numbers_1, '|') as phone_number_1,
    array_to_string(distinct_clusters.cluster_phone_numbers_2, '|') as phone_number_2,
    array_to_string(distinct_clusters.cluster_types, '|') as type,
    b.name as blkgrp_name,
    b.tract_name as tract_name,
    b.county_name as county_name,
    b.state_name as state_name,
    b.state_code as state_code,
    b.id as blkgrp_id,
    b.tract_id as tract_id,
    b.county_id as county_id,
    b.state_id as state_id,
    b.division_id as division_id,
    b.region_id as region_id,
    false as duplicate,
    st_centroid(st_union(distinct_clusters.cluster_geometries)) as geometry
from deduped_clusters
join distinct_clusters
    on deduped_clusters.cluster_id = distinct_clusters.cluster_id
join blkgrps b
    on st_contains(b.geometry, st_centroid(st_union(distinct_clusters.cluster_geometries)))

Running query in 'postgresql://healthgps:***@db/healthgps'

4786 rows affected.

++
||
++
++

In [39]:
%%sql
-- Set all those individual points as duplicates
UPDATE food_outposts fo
SET duplicate = TRUE
FROM (
    SELECT
        DISTINCT uu.uuid::uuid
    FROM food_outposts_clusters
    CROSS JOIN LATERAL unnest(string_to_array(alt_id, '|')) AS uu(uuid)
) dup
where fo.id = dup.uuid;

Running query in 'postgresql://healthgps:***@db/healthgps'

10154 rows affected.

++
||
++
++

In [41]:
%%sql
-- Filter out duplicates through a view
-- DROP VIEW food_outposts_processed;

CREATE VIEW
  food_outposts_processed AS
SELECT
  *
FROM
  food_outposts
WHERE
  duplicate = False

UNION ALL

SELECT
    *
FROM
    food_outposts_clusters;

Running query in 'postgresql://healthgps:***@db/healthgps'

++
||
++
++

In [76]:
%%sql
-- SNAP retailers
DROP TABLE snap_retailers;

CREATE TABLE
  snap_retailers (
    id UUID PRIMARY KEY,
    source VARCHAR(255),
    alt_id VARCHAR(255),
    name VARCHAR(255),
    address VARCHAR(255),
    type VARCHAR(255),
    blkgrp_name VARCHAR(255),
    tract_name VARCHAR(255),
    county_name VARCHAR(255),
    state_name VARCHAR(255),
    state_code VARCHAR(2),
    blkgrp_id VARCHAR(12),
    tract_id VARCHAR(11),
    county_id VARCHAR(5),
    state_id VARCHAR(2),
    division_id VARCHAR(1),
    region_id VARCHAR(1),
    geometry GEOMETRY (POINT, 4326)
  );

INSERT INTO
  snap_retailers (
    id,
    source,
    alt_id,
    name,
    address,
    type,
    blkgrp_name,
    tract_name,
    county_name,
    state_name,
    state_code,
    blkgrp_id,
    tract_id,
    county_id,
    state_id,
    division_id,
    region_id,
    geometry
  )
SELECT
  traceid AS id,
  'usda' AS source,
  snap."Record_ID" AS alt_id,
  UPPER(snap."Store_Name") AS name,
  UPPER(
    concat_ws (
      ', ',
      "Store_Street_Address",
      "City",
      "State",
      "Zip_Code"
    )
  ) AS address,
  UPPER(snap."Store_Type") AS type,
  blkgrps.name AS blkgrp_name,
  blkgrps.tract_name AS tract_name,
  blkgrps.county_name AS county_name,
  blkgrps.state_name AS state_name,
  blkgrps.state_code AS state_code,
  blkgrps.id AS blkgrp_id,
  blkgrps.tract_id AS tract_id,
  blkgrps.county_id AS county_id,
  blkgrps.state_id AS state_id,
  blkgrps.division_id AS division_id,
  blkgrps.region_id AS region_id,
  snap.geometry
FROM
  blkgrps
  INNER JOIN snap_retailers_raw snap ON st_contains (blkgrps.geometry, snap.geometry)

Running query in 'postgresql://healthgps:***@db/healthgps'

261966 rows affected.

++
||
++
++

In [90]:
# -- %%sql
# -- Create indices (RUN THESE IN SQL TERMINAL)
# -- CREATE INDEX states_geom_idx ON states USING GIST (geometry);
# -- CREATE INDEX counties_geom_idx ON counties USING GIST (geometry);
# -- CREATE INDEX tracts_geom_idx ON tracts USING GIST (geometry);
# -- CREATE INDEX blkgrps_geom_idx ON blkgrps USING GIST (geometry);
# -- CREATE INDEX food_posts_geom_idx ON food_posts USING GIST (geometry);
# -- CREATE INDEX snap_retailers_geom_idx ON snap_retailers USING GIST (geometry);

In [97]:
%%sql
-- Create analytical county table
DROP TABLE counties_metrics;
CREATE TABLE counties_metrics (
    id VARCHAR(5) PRIMARY KEY,
    ct_population INTEGER,
    ct_households INTEGER,
    ct_households_any_food_assistance INTEGER,
    ct_households_snap_food_assistance INTEGER,
    ct_households_snap_food_assistance_disability INTEGER,
    ct_housingunits INTEGER,
    ct_housingunits_owner_occupied INTEGER,
    ct_housingunits_owner_occupied_no_vehicle INTEGER,
    ct_housingunits_renter_occupied INTEGER,
    ct_housingunits_renter_occupied_no_vehicle INTEGER,
    ct_poverty_determined_population INTEGER,
    ct_poverty_income_ratio_lt_50pct INTEGER,
    ct_poverty_income_ratio_lte_99pct INTEGER,
    ct_poverty_income_ratio_lte_124pct INTEGER,
    ct_poverty_income_ratio_lte_149pct INTEGER,
    ct_poverty_income_ratio_lte_184pct INTEGER,
    ct_poverty_income_ratio_lte_199pct INTEGER,
    ct_poverty_income_ratio_gt_200pct INTEGER,
    ct_income_below_poverty INTEGER,
    ct_income_below_poverty_lt_6yo INTEGER,
    ct_income_below_poverty_lte_11yo INTEGER,
    ct_income_below_poverty_lte_17yo INTEGER,
    ct_snap_retailers INTEGER,
    ct_food_outposts INTEGER,
    fp VARCHAR(3),
    name VARCHAR(255),
    state_name VARCHAR(255),
    state_code VARCHAR(2),
    state_id VARCHAR(2),
    division_id VARCHAR(1),
    region_id VARCHAR(1)
);


INSERT INTO counties_metrics (
    id,
    ct_population,
    ct_households,
    ct_households_any_food_assistance,
    ct_households_snap_food_assistance,
    ct_households_snap_food_assistance_disability,
    ct_housingunits,
    ct_housingunits_owner_occupied,
    ct_housingunits_owner_occupied_no_vehicle,
    ct_housingunits_renter_occupied,
    ct_housingunits_renter_occupied_no_vehicle,
    ct_poverty_determined_population,
    ct_poverty_income_ratio_lt_50pct,
    ct_poverty_income_ratio_lte_99pct,
    ct_poverty_income_ratio_lte_124pct,
    ct_poverty_income_ratio_lte_149pct,
    ct_poverty_income_ratio_lte_184pct,
    ct_poverty_income_ratio_lte_199pct,
    ct_poverty_income_ratio_gt_200pct,
    ct_income_below_poverty,
    ct_income_below_poverty_lt_6yo,
    ct_income_below_poverty_lte_11yo,
    ct_income_below_poverty_lte_17yo,
    ct_snap_retailers,
    ct_food_outposts,
    fp,
    name,
    state_name,
    state_code,
    state_id,
    division_id,
    region_id
)
WITH agg AS (
    SELECT
        cty.id,
        count(distinct snap.id) as ct_snap_retailers,
        count(distinct outpost.id) as ct_food_outposts
    from counties cty
    LEFT JOIN snap_retailers snap
        ON st_contains(cty.geometry, snap.geometry)
    LEFT JOIN food_outposts outpost
        ON st_contains(cty.geometry, outpost.geometry)
    GROUP BY 1
)
SELECT
    cty.id,
    survey."ASN1E001"::integer as ct_population,
    survey."ASRAE001"::integer as ct_households,
    survey."ASRAE002"::integer as ct_households_any_food_assistance,
    survey."ASSKE002"::integer as ct_households_snap_food_assistance,
    survey."ASSKE003"::integer as ct_households_snap_food_assistance_disability,
    survey."ASUTE002"::integer as ct_housingunits,
    survey."ASUTE002"::integer as ct_housingunits_owner_occupied,
    survey."ASUTE003"::integer as ct_housingunits_owner_occupied_no_vehicle,
    survey."ASUTE009"::integer as ct_housingunits_renter_occupied,
    survey."ASUTE010"::integer as ct_housingunits_renter_occupied_no_vehicle,
    survey."ASQIE001"::integer as ct_poverty_determined_population,
    survey."ASQIE002"::integer as ct_poverty_income_ratio_lt_50pct,
    survey."ASQIE003"::integer as ct_poverty_income_ratio_lte_99pct,
    survey."ASQIE004"::integer as ct_poverty_income_ratio_lte_124pct,
    survey."ASQIE005"::integer as ct_poverty_income_ratio_lte_149pct,
    survey."ASQIE006"::integer as ct_poverty_income_ratio_lte_184pct,
    survey."ASQIE007"::integer as ct_poverty_income_ratio_lte_199pct,
    survey."ASQIE008"::integer as ct_poverty_income_ratio_gt_200pct,
    survey."AS7TE002"::integer as ct_income_below_poverty,
    survey."AS7TE003"::integer as ct_income_below_poverty_lt_6yo,
    survey."AS7TE004"::integer as ct_income_below_poverty_lte_11yo,
    survey."AS7TE005"::integer as ct_income_below_poverty_lte_17yo,
    agg.ct_snap_retailers as ct_snap_retailers,
    agg.ct_food_outposts as ct_food_outposts,
    cty.fp,
    cty.name,
    cty.state_name,
    cty.state_code,
    cty.state_id,
    cty.division_id,
    cty.region_id
from counties cty
INNER JOIN "acs_2023_county_raw" survey
    ON cty.id = survey."TL_GEO_ID"
LEFT JOIN agg
    ON cty.id = agg.id

Running query in 'postgresql://healthgps:***@db/healthgps'

3222 rows affected.

++
||
++
++

In [98]:
%%sql
-- Extend county_agg table with metrics from Feeding America
ALTER TABLE counties_metrics
    ADD COLUMN IF NOT EXISTS fa_high_income_threshold DECIMAL,
    ADD COLUMN IF NOT EXISTS fa_low_income_threshold DECIMAL,
    ADD COLUMN IF NOT EXISTS fa_high_income_pct DECIMAL,
    ADD COLUMN IF NOT EXISTS fa_between_income_pct DECIMAL,
    ADD COLUMN IF NOT EXISTS fa_low_income_pct DECIMAL,
    ADD COLUMN IF NOT EXISTS fa_low_income_threshold_programs TEXT,
    ADD COLUMN IF NOT EXISTS fa_high_income_threshold_programs TEXT,
    ADD COLUMN IF NOT EXISTS fa_pop NUMERIC,
    ADD COLUMN IF NOT EXISTS fa_fi_rate DECIMAL,
    ADD COLUMN IF NOT EXISTS fa_fi_pop NUMERIC,
    ADD COLUMN IF NOT EXISTS fa_child_fi_rate DECIMAL,
    ADD COLUMN IF NOT EXISTS fa_child_fi_pop NUMERIC,
    ADD COLUMN IF NOT EXISTS fa_child_fi_below_pct DECIMAL,
    ADD COLUMN IF NOT EXISTS fa_child_fi_above_pct DECIMAL,
    ADD COLUMN IF NOT EXISTS fa_cpm DECIMAL,
    ADD COLUMN IF NOT EXISTS fa_annual_gap_funding_needed NUMERIC,
    ADD COLUMN IF NOT EXISTS fa_associated_bank_ids TEXT[];


UPDATE counties_metrics c
SET
    fa_high_income_threshold = (x.finding->>'HI_TH')::DECIMAL,
    fa_low_income_threshold = (x.finding->>'LOW_TH')::DECIMAL,
    fa_high_income_pct = (x.finding->>'HI_PCT')::DECIMAL,
    fa_between_income_pct = (x.finding->>'BTWN_PCT')::DECIMAL,
    fa_low_income_pct = (x.finding->>'LOW_PCT')::DECIMAL,
    fa_low_income_threshold_programs = x.finding->>'LOW_TH_PROGS',
    fa_high_income_threshold_programs = x.finding->>'HI_TH_PROGS',
    fa_pop = (x.finding->>'COUNTY_POP')::NUMERIC,
    fa_fi_rate = (x.finding->>'COUNTY_FI_RATE')::DECIMAL,
    fa_fi_pop = (x.finding->>'COUNTY_POP_FI')::NUMERIC,
    fa_child_fi_rate = (x.finding->>'CHILD_FI_PCT')::DECIMAL,
    fa_child_fi_pop = (x.finding->>'CHILD_FI_COUNT')::NUMERIC,
    fa_child_fi_below_pct = (x.finding->>'CHILD_FI_BELOW_PCT')::DECIMAL,
    fa_child_fi_above_pct = (x.finding->>'CHILD_FI_ABOVE_PCT')::DECIMAL,
    fa_cpm = (x.finding->>'COST_PER_MEAL')::DECIMAL,
    fa_annual_gap_funding_needed = (x.finding->>'WT_ANNUAL_DOLLARS')::NUMERIC,
    fa_associated_bank_ids = x.fa_associated_bank_ids
FROM (
    WITH parsed_capacity AS (
        SELECT
            "EntityID" as id,
                REPLACE(
                    REPLACE(
                        REPLACE(
                            REPLACE(
                                REPLACE(
                                    REPLACE(
                                        REPLACE("ListFipsCounty", ' ', ''),
                                        ''',','",'
                                    ),
                                    ''':','":'
                                ),
                                ':''',
                                ':"'
                            ),
                            ',''',
                            ',"'
                        ),
                        '{''',
                        '{"'
                    ),
                    '''}',
                    '"}'
                ) as cleaned_json_text
        FROM feeding_america_foodbanks_raw
    ),
    county_arrays as (
        select
            id,
            cleaned_json_text::jsonb -> 'LocalFindings' AS local_findings
        from parsed_capacity
    )
    select
        elem->>'FipsCode' as county_id,
        elem AS finding,
        array_agg(id) as fa_associated_bank_ids
    from
    county_arrays
    cross join lateral (
        select jsonb_array_elements(
            case
                when local_findings is null then '[]'::jsonb
                when jsonb_typeof(local_findings) = 'array' then local_findings
                else jsonb_build_array(local_findings)
            end
        ) as elem
    )
    group by 1,2
) x
WHERE c.id = x.county_id;

Running query in 'postgresql://healthgps:***@db/healthgps'

3144 rows affected.

++
||
++
++

In [100]:
%%sql
-- Add in metrics to filter out on poverty lines
ALTER TABLE counties_metrics
    -- DROP COLUMN filter_step_metric,
    -- DROP COLUMN filter_step_bucket,
    ADD COLUMN IF NOT EXISTS  filter_step_metric DECIMAL,
    ADD COLUMN IF NOT EXISTS filter_step_bucket INTEGER;

UPDATE counties_metrics c
SET
    filter_step_metric = x.z_metric,
    filter_step_bucket = x.z_metric_bucket
FROM (
    WITH init AS (
        SELECT
            id,
            name,
            state_id,
            state_code,
            ct_income_below_poverty,
            ct_poverty_determined_population
        FROM counties_metrics
        WHERE state_code != 'PR'
    ),
    metrics AS (
        SELECT
            id,
            name,
            state_id,
            state_code,
            1.0 * ct_income_below_poverty / ct_poverty_determined_population as metric
        FROM init
    ),
    stats AS (
        SELECT
            avg(metric) as avg_metric,
            stddev_pop(metric) as sd_metric
        FROM metrics
    )
    SELECT
        m.id as id,
        m.name,
        m.state_code,
        m.metric as z_metric,
        CASE
            WHEN abs((m.metric - s.avg_metric) / s.sd_metric) BETWEEN -1 AND 1 THEN 1
            WHEN ((m.metric - s.avg_metric) / s.sd_metric) BETWEEN -2 AND -1 THEN -2
            WHEN ((m.metric - s.avg_metric) / s.sd_metric) BETWEEN 1 AND 2 THEN 2
            WHEN ((m.metric - s.avg_metric) / s.sd_metric) BETWEEN -3 AND -2 THEN -3
            WHEN ((m.metric - s.avg_metric) / s.sd_metric) BETWEEN 2 AND 3 THEN 3
            WHEN ((m.metric - s.avg_metric) / s.sd_metric) < -3 THEN -4
            WHEN ((m.metric - s.avg_metric) / s.sd_metric) > 3 THEN 4
            ELSE 1000
        END AS z_metric_bucket
    FROM metrics m
    CROSS JOIN stats s
) x
WHERE c.id = x.id;

Running query in 'postgresql://healthgps:***@db/healthgps'

3144 rows affected.

++
||
++
++

In [ ]:
# INGEST census_cps_fss_raw VIA PANDAS

In [134]:
%%sql
-- Include survey "Yes" response metrics
-- DROP VIEW counties_analytics_view;
ALTER TABLE counties_metrics
    DROP COLUMN coverage_step_metric,
    -- DROP COLUMN coverage_step_contrib,
    ADD COLUMN IF NOT EXISTS coverage_step_metric INTEGER
    -- ADD COLUMN IF NOT EXISTS coverage_step_contrib DECIMAL;
    ;

UPDATE counties_metrics c
SET
    coverage_step_metric = x.vicinity_survey_response_count
FROM (
    select
        lpad("GESTFIPS"::text,2,'0') || lpad("GTCO"::text, 3, '0') as county_id,
        count(*) FILTER (WHERE "HESC3" = 1) as utility_survey_response_count,
        count(*) FILTER (WHERE "HESC3A" = 1) as vicinity_survey_response_count
    from census_cps_fss_raw
        group by 1
) x
where c.id = x.county_id

Running query in 'postgresql://healthgps:***@db/healthgps'

276 rows affected.

++
||
++
++

In [138]:
# Run studentized residuals for outlier detection

import pandas
query = """
select
    state_id,
    state_code,
    sum(ct_food_outposts) as y,
    sum(ct_households_any_food_assistance) as x,
    sum(coverage_step_metric) as x_bk,
    1.0 * sum(ct_food_outposts) / sum(ct_population) as y_alt,
    1.0 * sum(coverage_step_metric) / sum(ct_population) as x_alt
from counties_metrics
group by 1,2
"""
import statsmodels.api as sm
from statsmodels.formula.api import ols

df = pandas.read_sql(query, con=engine)
model = ols('x ~ y', data=df).fit()
stud_res = model.outlier_test()
df = df.join(stud_res)
df["coverage_step_contrib"] = df["student_resid"]

In [136]:
# counties_metrics = pandas.read_sql("select * from counties_metrics", con=engine)
# combined_df = pandas.merge(counties_metrics, df[["state_id", "coverage_step_contrib"]], on="state_id", how="left")
# combined_df.to_sql("counties_metrics", con=engine, if_exists="replace", index=False)

252

In [139]:
df

,state_id,state_code,y,x,x_bk,y_alt,x_alt,student_resid,unadj_p,bonf(p),coverage_step_contrib
0,47,TN,936.0,328525.0,313.0,1.339807e-04,0.000045,0.151200,0.880439,1.000000,0.151200
1,09,CT,493.0,182048.0,NaN,1.370073e-04,NaN,0.216815,0.829253,1.000000,0.216815
2,13,GA,1887.0,523771.0,312.0,1.743575e-04,0.000029,-0.737871,0.464112,1.000000,-0.737871
3,34,NJ,1129.0,342464.0,627.0,1.218300e-04,0.000068,-0.186337,0.852950,1.000000,-0.186337
4,40,OK,974.0,224347.0,NaN,2.437889e-04,NaN,-0.579914,0.564630,1.000000,-0.579914
5,11,DC,92.0,45643.0,624.0,1.368887e-04,0.000928,0.253781,0.800727,1.000000,0.253781
6,33,NH,216.0,39398.0,530.0,1.556382e-04,0.000382,-0.060222,0.952224,1.000000,-0.060222
7,28,MS,566.0,162050.0,NaN,1.917709e-04,NaN,-0.068307,0.945819,1.000000,-0.068307
8,36,NY,3653.0,1252826.0,1240.0,1.838235e-04,0.000062,-0.094417,0.925163,1.000000,-0.094417
9,50,VT,337.0,31549.0,NaN,5.222749e-04,NaN,-0.376573,0.708117,1.000000,-0.376573


In [151]:
df = %sql select * from counties_analytics_view

Running query in 'postgresql://healthgps:***@db/healthgps'

3222 rows affected.

In [154]:
df

id,ct_population,ct_households,ct_households_any_food_assistance,ct_households_snap_food_assistance,ct_households_snap_food_assistance_disability,ct_housingunits,ct_housingunits_owner_occupied,ct_housingunits_owner_occupied_no_vehicle,ct_housingunits_renter_occupied,ct_housingunits_renter_occupied_no_vehicle,ct_poverty_determined_population,ct_poverty_income_ratio_lt_50pct,ct_poverty_income_ratio_lte_99pct,ct_poverty_income_ratio_lte_124pct,ct_poverty_income_ratio_lte_149pct,ct_poverty_income_ratio_lte_184pct,ct_poverty_income_ratio_lte_199pct,ct_poverty_income_ratio_gt_200pct,ct_income_below_poverty,ct_income_below_poverty_lt_6yo,ct_income_below_poverty_lte_11yo,ct_income_below_poverty_lte_17yo,ct_snap_retailers,ct_food_outposts,fp,name,state_name,state_code,state_id,division_id,region_id,fa_high_income_threshold,fa_low_income_threshold,fa_high_income_pct,fa_between_income_pct,fa_low_income_pct,fa_low_income_threshold_programs,fa_high_income_threshold_programs,fa_pop,fa_fi_rate,fa_fi_pop,fa_child_fi_rate,fa_child_fi_pop,fa_child_fi_below_pct,fa_child_fi_above_pct,fa_cpm,fa_annual_gap_funding_needed,fa_associated_bank_ids,filter_step_metric,filter_step_bucket,coverage_step_contrib_x,coverage_step_contrib_y,coverage_step_metric,coverage_step_contrib,pct_snap_retailers,pct_metric,geometry
31039,8976,3782,189,165,65,2695,2695,44,1087,38,8841,311,375,411,424,405,352,6563,686,79,52,102,8,2,039,CUMING,NEBRASKA,NE,31,4,2,1.85,1.65,0.49,0.0,0.51,SNAP,OtherNutritionProgram,8976.0,0.129,1160.0,0.185,410.0,0.6,0.4,3.46,761000.0,{59},0.07759303246239113,-2.0,-0.2758113925505376,-0.13224720505660356,None,-0.13224720505660412,80.0000000000000000,0.0620744259699129057760000000000000000000,...
53069,4573,2026,139,119,78,1718,1718,32,308,49,4543,345,369,162,290,715,15,2647,714,31,69,54,5,3,069,WAHKIAKUM,WASHINGTON,WA,53,9,4,2.0,2.0,0.13,0.0,0.87,"SNAP,OtherNutritionPrograms","SNAP,OtherNutritionPrograms",4573.0,0.162,740.0,0.233,200.0,0.79,0.21,3.75,526000.0,{74},0.15716486902927582,1.0,-0.9429087230901556,-0.83357369805603,None,-0.8335736980560304,62.5000000000000000,0.0982280431432973805875000000000000000000,...
35011,1580,727,173,172,83,502,502,0,225,33,1566,92,230,25,26,202,43,948,322,8,57,33,5,3,011,DE BACA,NEW MEXICO,NM,35,8,4,2.0,2.0,0.28,0.0,0.72,"SNAP,OtherNutritionPrograms","SNAP,OtherNutritionPrograms",1580.0,0.183,290.0,0.196,70.0,0.76,0.24,3.52,193000.0,{1},0.20561941251596424,2.0,0.7504836098154977,0.5773057147511479,None,0.5773057147511472,62.5000000000000000,0.1285121328224776500625000000000000000000,...
31109,323673,132162,12210,11086,5388,77767,77767,1487,54395,5930,310515,17723,18548,11184,13438,18060,8087,223475,36271,2856,2625,2888,180,18,109,LANCASTER,NEBRASKA,NE,31,4,2,1.85,1.65,0.47,0.0,0.53,SNAP,OtherNutritionProgram,323673.0,0.146,47380.0,0.195,14330.0,0.65,0.35,3.74,33579000.0,{126},0.11680917185965252,1.0,-0.2758113925505376,-0.13224720505660356,None,-0.13224720505660412,90.9090909090909091,0.1061901562360477388728334628835087750116,...
31129,4089,1819,81,80,59,1482,1482,1,337,62,4035,170,130,164,143,397,108,2923,300,39,16,19,3,0,129,NUCKOLLS,NEBRASKA,NE,31,4,2,1.85,1.65,0.48,0.0,0.52,SNAP,OtherNutritionProgram,4089.0,0.13,530.0,0.195,150.0,0.79,0.21,3.3,331000.0,{59},0.07434944237918216,-2.0,-0.2758113925505376,-0.13224720505660356,None,-0.13224720505660412,100.0000000000000000,0.0743494423791821561300000000000000000000,...
72085,35034,12519,6725,6713,2165,9710,9710,863,2809,335,34920,5421,7690,3312,3790,2694,1447,10566,13111,710,1301,933,0,0,085,LAS PIEDRAS,PUERTO RICO,PR,72,0,9,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,...
46099,200689,82394,7780,7225,2773,52328,52328,797,30066,3584,195878,7558,9758,7534,5887,12574,4770,147797,17316,2051,1796,1461,181,5,099,MINNEHAHA,SOUTH DAKOTA,SD,46,4,2,1.85,1.3,0.61,0.0,0.39,SNAP,OtherNutritionProgram,200689.0,0.107,21520.0,0.166,8520.0,0.68,0.32,3.78,15421000.0,{144},0.08840196448809974,1.0,No